## Setup

```bash
pip install pixeltable google-genai 'fastapi[standard]'
pip install torch transformers  # optional, for object detection
```

Set `GEMINI_API_KEY` as an environment variable before launching the notebook.

In [ ]:
import pixeltable as pxt
from pixeltable.functions import gemini

BASE_URL = 'https://raw.githubusercontent.com/pixeltable/pixeltable/release/docs/resources'

## Storage

Create a table with native video and string column types. No file wrangling, no S3 buckets.

In [ ]:
pxt.drop_dir('demo', force=True, if_not_exists='ignore')
pxt.create_dir('demo')

videos = pxt.create_table('demo/videos', {'video': pxt.Video, 'title': pxt.String})
videos

## Orchestration

Add computed columns that run automatically on every insert. Gemini analyzes each video and produces a text description.

In [ ]:
videos.add_computed_column(
    response=gemini.generate_content(
        [videos.video, 'Describe this video in detail.'], model='gemini-3-flash-preview'
    )
)

videos.add_computed_column(
    description=videos.response.candidates[0].content.parts[0].text.astype(pxt.String)
)

videos.add_embedding_index('description', embedding=gemini.embed_content.using(model='gemini-embedding-2-preview'))

## Insert

Inserting rows triggers the full pipeline: download, Gemini analysis, text extraction, embedding. Open the **Dashboard** to watch progress.

In [ ]:
videos.insert([
    {'video': f'{BASE_URL}/bangkok.mp4', 'title': 'Bangkok Street Tour'},
    {'video': f'{BASE_URL}/The-Pursuit-of-Happiness-Video-Extract.mp4', 'title': 'The Pursuit of Happiness'},
])

In [ ]:
videos = pxt.get_table('demo/videos')
videos.select(videos.title, videos.description).collect()

## Retrieval: Semantic Search

Search by meaning, not keywords. The embedding index makes this a single line.

In [ ]:
sim = videos.description.similarity(string='street food')
videos.order_by(sim, asc=False).limit(5).select(videos.title, sim).collect()

## Retrieval: On-the-Fly Object Detection

Extract a frame and run HuggingFace DETR at query time. Nothing is pre-computed; the model runs on the fly.

In [ ]:
from pixeltable.functions import huggingface

videos.select(
    videos.title,
    detections=huggingface.detr_for_object_detection(
        videos.video.extract_frame(timestamp=2.0), model_id='facebook/detr-resnet-50',
    ),
).collect()

## Serving

Expose queries and inserts as HTTP endpoints with zero boilerplate. In production, use `pxt serve service.toml`.

In [ ]:
import json
import fastapi
from fastapi.testclient import TestClient
from pixeltable.serving import FastAPIRouter

@pxt.query
def search_videos(query_text: str, limit: int = 5):
    sim = videos.description.similarity(string=query_text)
    return videos.order_by(sim, asc=False).limit(limit).select(videos.title, videos.description, sim)

app = fastapi.FastAPI()
router = FastAPIRouter()
router.add_query_route(path='/search', query=search_videos)
router.add_insert_route(videos, path='/ingest', inputs=['video', 'title'], outputs=['title', 'description'])
app.include_router(router)

client = TestClient(app)

In [ ]:
resp = client.post('/search', json={'query_text': 'street food', 'limit': 2})
resp.json()

In [ ]:
resp = client.post('/ingest', json={'video': f'{BASE_URL}/bangkok.mp4', 'title': 'Test Upload'})
resp.json()